<a href="https://colab.research.google.com/github/yagav-alt/text-to-sql-prompt-pipeline/blob/main/txt_to_sql_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-genai

In [ ]:
import os
from google import genai
from google.genai import types
from google.colab import userdata

# Automatically fetches the secret key you just saved securely
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=API_KEY)
except Exception as e:
    print("Make sure you added GEMINI_API_KEY to your Colab Secrets (key icon on the left panel)!")

# 1. Define your Database Schema
DB_SCHEMA = """
You are a database assistant. Generate standard SQL based on this schema:

Table: employees
Columns:
  - employee_id (INT, Primary Key)
  - first_name (VARCHAR)
  - last_name (VARCHAR)
  - department_id (INT)
  - salary (INT)
  - status (VARCHAR) -- 'Active' or 'Inactive'

Table: departments
Columns:
  - department_id (INT, Primary Key)
  - department_name (VARCHAR)
"""

# 2. Define the Strict Prompt Engineering Rules
PROMPT_RULES = """
CRITICAL SYNTAX RULES:
1. For "not equal to" operations, you MUST use the `!=` operator. Never use `<>`.
2. For range filtering, explicitly use `>=` and `<=`. Do not use the `BETWEEN` operator.
3. Return ONLY the raw SQL code. Do not include markdown code blocks (```sql), explanation, or conversational text.
"""

def generate_sql(user_question: str) -> str:
    """Sends the question to Gemini with system instructions to get clean SQL."""
    system_instruction = f"{DB_SCHEMA}\n{PROMPT_RULES}"

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f"User Question: {user_question}",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.0,
        ),
    )
    return response.text.strip()

# ---- Test It Out ----
test_question = "Show me the first and last names of active employees who do not work in department 10 and earn more than 50000"

generated_query = generate_sql(test_question)
print("--- Generated SQL ---")
print(generated_query)

--- Generated SQL ---
SELECT first_name, last_name
FROM employees
WHERE status = 'Active' AND department_id != 10 AND salary > 50000


In [ ]:
# List of diverse test questions to evaluate our prompt's reliability
test_cases = [
    {
        "id": 1,
        "question": "List all active employees who earn between 40000 and 80000.",
        "expected_rule": "Must use >= and <=, NOT the BETWEEN operator."
    },
    {
        "id": 2,
        "question": "Show details of all departments that are not department 5.",
        "expected_rule": "Must use != operator, NOT <>."
    },
    {
        "id": 3,
        "question": "Find the first name of inactive employees in department 20.",
        "expected_rule": "Must filter by status = 'Inactive'."
    }
]

print("=== STARTING PROMPT ENGINEERING EVALUATION PIPELINE ===\n")

# Loop through our test suite to check for consistency
for case in test_cases:
    print(f"Test Case #{case['id']}")
    print(f"User Request: {case['question']}")
    print(f"Target Constraint: {case['expected_rule']}")

    # Run our prompt engine
    generated_sql_output = generate_sql(case['question'])

    print("Generated Output:")
    print(generated_sql_output)
    print("-" * 50)

=== STARTING PROMPT ENGINEERING EVALUATION PIPELINE ===

Test Case #1
User Request: List all active employees who earn between 40000 and 80000.
Target Constraint: Must use >= and <=, NOT the BETWEEN operator.
Generated Output:
SELECT employee_id, first_name, last_name, department_id, salary, status
FROM employees
WHERE status = 'Active' AND salary >= 40000 AND salary <= 80000;
--------------------------------------------------
Test Case #2
User Request: Show details of all departments that are not department 5.
Target Constraint: Must use != operator, NOT <>.
Generated Output:
SELECT *
FROM departments
WHERE department_id != 5
--------------------------------------------------
Test Case #3
User Request: Find the first name of inactive employees in department 20.
Target Constraint: Must filter by status = 'Inactive'.
Generated Output:
SELECT first_name
FROM employees
WHERE status = 'Inactive' AND department_id = 20
--------------------------------------------------
